# The recipe as cooked: what do home cooks change, and where?

Review analysis for the 02805 project.

**Before running:** this notebook loads the cleaned data made by `project_Rakel.ipynb`. Run that notebook once first (it creates the folder `data/`), and again if the preprocessing there is changed.

In [ ]:
import numpy as np
import pandas as pd

recipes = pd.read_parquet("data/recipes_clean.parquet")
reviews = pd.read_parquet("data/reviews_clean.parquet")
communities = pd.read_parquet("data/communities.parquet")

# parquet gives lists back as numpy arrays, so turn them into normal Python lists again
for col in ["ingredients", "ingr_clean", "tags", "steps", "nutrition"]:
    recipes[col] = recipes[col].apply(list)

print(f"Recipes:     {len(recipes):,}")
print(f"Reviews:     {len(reviews):,}")
print(f"Communities: {communities['community'].nunique()} (broad: {communities['community_broad'].nunique()})")

In [ ]:
reviews.head()

## Test run: can the reviews be used?

Before building anything, we check four things:

1. What do the texts look like, and how long are they?
2. How many reviews mention a change to the recipe?
3. Can we pull out *"X instead of Y"* and match both X and Y to our clean ingredients?
4. A first look at the hypothesis: do people change baking recipes less?

### 1. The texts

The reviews contain leftovers from the website's HTML, e.g. `&#039;` for an apostrophe and `<br/>` for a line break. We turn these back into normal text and lowercase everything (week 5).

In [ ]:
import html
import re

print("With HTML codes (&...;):", reviews["review"].str.contains(r"&\w+;|&#\d+;").sum())
print("With <br/>:", reviews["review"].str.contains("<br").sum())

def clean_text(text):
    text = html.unescape(html.unescape(text))   # twice, because some are encoded twice (&amp;#039;)
    text = re.sub(r"<[^>]+>", " ", text)          # remove <br/> and other tags
    return " ".join(text.lower().split())         # lowercase, one space between words

reviews["text"] = reviews["review"].apply(clean_text)
reviews["n_words"] = reviews["text"].str.split().str.len()

print(reviews["n_words"].describe().round(0))

### 2. How many reviews mention a change?

A first, simple measurement: does the review contain words that people use when they change a recipe? This will miss some changes and wrongly catch others (e.g. the question *"has anyone tried a replacement for bread?"* is caught by *replace*), so we check it by hand later.

In [ ]:
CHANGE_WORDS = {
    "instead of": r"\binstead of\b",
    "substitute": r"\bsubstitut",
    "in place of": r"\bin place of\b",
    "replace": r"\breplac",
    "added / I add": r"\badded\b|\bi add\b",
    "left out / omitted": r"\bleft out\b|\bomit|\bleave out\b",
    "more / less / extra": r"\b(?:more|less|extra)\b",
    "as written / no changes": r"as written|exactly as|no changes|wouldn't change|would not change",
}

for label, pattern in CHANGE_WORDS.items():
    print(f"{label:25s} {reviews['text'].str.contains(pattern).mean():6.1%}")

change_pattern = r"\binstead of\b|\bsubstitut|\bin place of\b|\breplac|\badded\b|\bi add\b|\bleft out\b|\bomit|\bleave out\b"
reviews["mentions_change"] = reviews["text"].str.contains(change_pattern)
print(f"\nReviews that mention a change (first 5 kinds): {reviews['mentions_change'].mean():.1%}")

### 3. Can we pull out *X instead of Y*?

We split each review into clauses at punctuation and at *and*, *but*, *so*, and look for *X instead of Y* inside each clause. Splitting first matters: in *"i use light margarine instead of butter and splenda sweetener instead of sugar"* there are two substitutions, and without splitting the second one was swallowed by the first (a first version of the code did that).

Then we look for our clean ingredient names (from `recipes["ingr_clean"]`) in X and in Y. If a piece contains several names, we take the longest (so *ground chicken* wins over *chicken*).

A strong automatic check: **Y (the ingredient that was replaced) should be in the recipe's own ingredient list.**

In [ ]:
vocabulary = sorted(set(x for xs in recipes["ingr_clean"] for x in xs), key=len, reverse=True)
vocab_pattern = re.compile(r"\b(" + "|".join(re.escape(v) for v in vocabulary) + r")s?\b")

def find_ingredient(piece):
    found = vocab_pattern.findall(piece)
    return max(found, key=len) if found else None

rows = []
for r in reviews[reviews["text"].str.contains("instead of")].itertuples():
    for clause in re.split(r"[.,!?;:()/]| and | but | so ", r.text):
        m = re.search(r"(.*) instead of (?:the |using |adding )?(.*)", clause)
        if m:
            x_text, y_text = m.group(1)[-40:], m.group(2)[:40]
            rows.append({"recipe_id": r.recipe_id, "context": clause.strip(),
                         "x_text": x_text, "y_text": y_text,
                         "x": find_ingredient(x_text), "y": find_ingredient(y_text)})

subs = pd.DataFrame(rows)
ingr_of = dict(zip(recipes["id"], recipes["ingr_clean"]))
subs["y_in_recipe"] = [y in ingr_of[r] for y, r in zip(subs["y"], subs["recipe_id"])]

both = subs.dropna(subset=["x", "y"])
both = both[both["x"] != both["y"]]
print(f"'instead of' found:              {len(subs):,}")
print(f"Both X and Y matched an ingredient: {len(both):,} ({len(both) / len(subs):.0%})")
print(f"... and Y is in the recipe:          {both['y_in_recipe'].sum():,} ({both['y_in_recipe'].mean():.0%} of those)")
print(f"Different substitution pairs:        {both[['x', 'y']].drop_duplicates().shape[0]:,}")

Examples, and the most common substitutions:

In [ ]:
for r in both.sample(12, random_state=1).itertuples():
    print(f"{r.x:20s} instead of {r.y:20s} | in recipe: {r.y_in_recipe!s:5s} | \"{r.x_text} instead of {r.y_text}\"")

In [ ]:
# most common substitutions: "x instead of y" (only where y is in the recipe)
both[both["y_in_recipe"]].groupby(["x", "y"]).size().sort_values(ascending=False).head(25)

### 4. A first look: do people change baking recipes less?

Share of reviews that mention a change, in the 3 broad communities. **Not tested yet**: this is only to see whether the idea is worth pursuing.

In [ ]:
reviews = reviews.merge(communities, left_on="recipe_id", right_on="id", how="left")

broad_names = {}
for c in sorted(communities["community_broad"].unique()):
    ids = communities.loc[communities["community_broad"] == c, "id"]
    top = pd.Series([x for xs in recipes.loc[recipes["id"].isin(ids), "ingr_clean"] for x in xs]).value_counts().index[:4]
    broad_names[c] = ", ".join(top)

summary = reviews.groupby("community_broad")["mentions_change"].agg(["mean", "size"])
summary.index = [broad_names[c] for c in summary.index]
summary.columns = ["share mentioning a change", "reviews"]
summary.round(3)

### What the test run shows

- **The texts are usable.** Median 43 words per review. About 19,000 contained HTML codes, now cleaned.
- **Changes are common.** 26.5 % of the reviews use one of the change words (*instead of, substitute, replace, added, left out*). 7.7 % contain *instead of*.
- **Substitutions can be extracted.** 27,837 clauses with *instead of*; in 11,130 both sides match one of our ingredients, and in 7,757 of those the replaced ingredient is really in the recipe. That gives about 4,100 different substitution pairs, enough for a substitution network.
- **The most common substitutions make sense:** butter instead of margarine, olive oil instead of butter, turkey instead of beef, chicken broth instead of water, honey instead of sugar, yogurt instead of sour cream.
- **Errors we can see:** *coconut water* becomes *coconut* (the name is not in our vocabulary), and *butterscotch pudding instead of vanilla [pudding]* loses the word *pudding* on the right side. The check *Y is in the recipe* removes many of these. A first version of the code also lost the second substitution when a review had two in a row (fixed by splitting into clauses). We need a hand-labelled sample to measure the precision.
- **First look at the hypothesis (not tested):** savoury 28.6 %, Asian-style 25.8 %, baking 23.5 % of reviews mention a change. The direction fits *baking is chemistry*, but we still need a null model, and to check that it is not just a few recipes or users driving it.

## Question 1: Do people change baking recipes less?

*Baking is chemistry, cooking is art*: if that is true, reviewers should change savoury recipes more than baking recipes.

### Step 1: share of changes per recipe

For every recipe: the share of its reviews that mention a change. We measure per **recipe**, not per review, because otherwise the most popular recipes would decide the result (one recipe has 1,609 reviews, most have 30–80).

In [ ]:
per_recipe = (reviews.groupby("recipe_id")
                     .agg(change_share=("mentions_change", "mean"), n_reviews=("mentions_change", "size"))
                     .reset_index()
                     .merge(communities, left_on="recipe_id", right_on="id"))

print(per_recipe["change_share"].describe().round(3))

name_of = dict(zip(recipes["id"], recipes["name"].str.split().str.join(" ")))
print("\nChanged most:")
for r in per_recipe.nlargest(5, "change_share").itertuples():
    print(f"  {r.change_share:.0%}  {name_of[r.recipe_id]}  ({r.n_reviews} reviews)")
print("Changed least:")
for r in per_recipe.nsmallest(5, "change_share").itertuples():
    print(f"  {r.change_share:.0%}  {name_of[r.recipe_id]}  ({r.n_reviews} reviews)")

### Step 2: compare the communities

We use the 13 Louvain communities from `project_Rakel.ipynb` (γ = 1.5). Each community is named after the ingredients that are most **over-represented** in it: the share of its recipes that use the ingredient, divided by the share of all recipes that use it (only ingredients used by at least 20 of its recipes, or by half of them in very small communities). The names are our interpretation, not the model's output.

In [ ]:
ingredient_sets = dict(zip(recipes["id"], recipes["ingr_clean"].apply(set)))
overall = pd.Series([x for xs in recipes["ingr_clean"] for x in xs]).value_counts() / len(recipes)

community_name = {}
for c, group in communities.groupby("community"):
    counts = pd.Series([x for i in group["id"] for x in ingredient_sets[i]]).value_counts()
    counts = counts[counts >= min(20, max(2, len(group) // 2))]   # small communities: used by at least half
    lift = (counts / len(group)) / overall[counts.index]
    community_name[c] = ", ".join(lift.nlargest(4).index)

by_community = (per_recipe.groupby("community")["change_share"]
                          .agg(["mean", "std", "size"])
                          .sort_values("mean"))
by_community["se"] = by_community["std"] / np.sqrt(by_community["size"])   # standard error of the mean
by_community["name"] = [community_name[c] for c in by_community.index]
by_community[["name", "size", "mean", "se"]].round(3)

In [ ]:
import matplotlib.pyplot as plt

overall_mean = per_recipe["change_share"].mean()

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh([f"{n[:45]}  ({s})" for n, s in zip(by_community["name"], by_community["size"])],
        by_community["mean"], xerr=by_community["se"], color="#0072B2", alpha=0.85)
ax.axvline(overall_mean, color="#D55E00", ls="--", label=f"All recipes ({overall_mean:.1%})")
ax.set_xlabel("Share of a recipe's reviews that mention a change (mean over recipes, ± standard error)")
ax.set_title("How often do reviewers change the recipe? By ingredient community")
ax.legend(frameon=False, loc="center right")
plt.tight_layout()
plt.show()

**What we see (not tested yet):**

- **Yeast bread is changed least**: 20.4 % of the reviews mention a change, against 26.6 % for all recipes. This fits *baking is chemistry*: if you change the yeast, flour or water, the bread may not rise.
- **Soups and stews (35 %) and Mexican food (33 %) are changed most**: dishes where you can add or swap vegetables, beans and spices without breaking anything.
- **But sweet baking is only a little below average** (cookies, cakes, chocolate: 24.8 %). So the picture is more nuanced than *baking vs cooking*: it is the *bread* that people do not touch.
- The small communities (5–67 recipes) have wide error bars, so we should not read much into them.

The null model and robustness checks below test whether this holds.

### Step 3: null model

Are the differences between communities larger than chance? We shuffle the community labels between the recipes 1,000 times (each recipe keeps its own share of changes, and each community keeps its size), and compute the community means again each time (week 8: *a text measurement placed on nodes is an attribute, tested with a shuffle*).

For each community we report the real mean, the range that 95 % of the shuffles fall in, and a z-score.

In [ ]:
rng = np.random.default_rng(0)
labels = per_recipe["community"].to_numpy()
real = per_recipe.groupby("community")["change_share"].mean()

null = pd.DataFrame([per_recipe["change_share"].groupby(rng.permutation(labels)).mean() for _ in range(1000)])

test = pd.DataFrame({"name": [community_name[c][:45] for c in real.index],
                     "recipes": per_recipe.groupby("community").size(),
                     "real": real,
                     "null_low": null.quantile(0.025),
                     "null_high": null.quantile(0.975),
                     "z": (real - null.mean()) / null.std()}).sort_values("real")
test.round(3)

One number for all communities together: the spread of the community means (their standard deviation, weighted by size). How often does a shuffle give a spread as large as the real one?

In [ ]:
sizes = per_recipe.groupby("community").size()


def spread(means):
    return np.sqrt(np.average((means - means.mean()) ** 2, weights=sizes[means.index]))


real_spread = spread(real)
null_spread = null.apply(spread, axis=1)
p = (1 + (null_spread >= real_spread).sum()) / (1 + len(null_spread))
print(f"Spread of community means, real: {real_spread:.4f}")
print(f"Spread in shuffles: {null_spread.mean():.4f} ± {null_spread.std():.4f}")
print(f"p-value: {p:.4f} (none of the 1,000 shuffles reached the real spread)" if p < 0.002 else f"p-value: {p:.3f}")

### Step 4: robustness — is it something boring?

Three simpler explanations we have to rule out (week 8, *Convincing: a comparison that rules out the boring explanation*):

1. **The word *added*.** *I added a little more salt* is a change, but *added* is also the broadest of our words. We repeat the analysis with a **strict** measure that only uses *instead of, substitute, in place of, replace, left out, omit*.
2. **Super-users.** A few users write thousands of reviews. If one of them always writes *I added...*, they could drive a whole community. We repeat without users who have more than 100 reviews in our data.
3. **Number of ingredients.** Bread has few ingredients, so maybe there is simply less to change. We compare recipes with the *same* number of ingredients.
4. **Our measure misses many changes.** The hand labels (see the end of the notebook) show that the change words find only about half of the reviews that describe a change: people often just write *I used frozen raspberries*, *I didn't use the oil* or *I doubled the oregano*. If bread bakers and soup cooks describe their changes with different words, this could create the differences. We therefore repeat the analysis with a **broad** measure that also catches these phrases.

In [ ]:
strict_pattern = r"\binstead of\b|\bsubstitut|\bin place of\b|\breplac|\bleft out\b|\bomit|\bleave out\b"
reviews["mentions_change_strict"] = reviews["text"].str.contains(strict_pattern)

# broad measure: also everyday phrases found in the hand-labelled sample
broad_pattern = (change_pattern
                 + r"|\bi used\b|\bwe used\b|\bused (?:a |some |fresh |frozen |dried |only |half |less |more |regular )"
                 + r"|\bdid ?n[o']?t (?:use|add|have|put)\b|\bwithout (?:the )?\w+"
                 + r"|\bdoubled\b|\bhalved\b|\breduced\b|\bcut (?:back|down)\b|\bskipped\b|\bi use (?:more|less)\b"
                 + r"|\b(?:i|we) (?:also |did )?(?:add|put|threw|tossed|sprinkled|mixed)\b|\bhad to add\b|\bexcept\b")
reviews["mentions_change_broad"] = reviews["text"].str.contains(broad_pattern)
print(f"Reviews that mention a change, broad measure: {reviews['mentions_change_broad'].mean():.1%}")

reviews_per_user = reviews["user_id"].value_counts()
super_users = reviews_per_user[reviews_per_user > 100].index
print(f"Users with more than 100 reviews: {len(super_users)}, writing {reviews['user_id'].isin(super_users).mean():.0%} of all reviews")


def community_means(data, column):
    shares = data.groupby("recipe_id")[column].mean()
    return shares.groupby(shares.index.map(dict(zip(communities["id"], communities["community"])))).mean()


robust = pd.DataFrame({
    "name": [community_name[c][:45] for c in real.index],
    "main measure": real,
    "strict measure": community_means(reviews, "mentions_change_strict"),
    "broad measure": community_means(reviews, "mentions_change_broad"),
    "without super-users": community_means(reviews[~reviews["user_id"].isin(super_users)], "mentions_change"),
}).sort_values("main measure")
print(robust.drop(columns="name").corr(method="spearman").round(2))
robust.round(3)

The number of ingredients: share of changes for recipes with the same number of clean ingredients, yeast bread against all other recipes.

In [ ]:
per_recipe["n_ingredients"] = per_recipe["recipe_id"].map(dict(zip(recipes["id"], recipes["ingr_clean"].str.len())))
bread = [c for c, n in community_name.items() if "yeast" in n][0]
per_recipe["group"] = np.where(per_recipe["community"] == bread, "yeast bread", "all other recipes")
per_recipe["ingredient_bin"] = pd.cut(per_recipe["n_ingredients"], [0, 5, 7, 9, 11, 50], labels=["1-5", "6-7", "8-9", "10-11", "12+"])

print(f"Correlation between number of ingredients and share of changes: "
      f"{per_recipe['n_ingredients'].corr(per_recipe['change_share'], method='spearman'):.2f} (Spearman)")
per_recipe.pivot_table(index="ingredient_bin", columns="group", values="change_share",
                       aggfunc=["mean", "size"], observed=True).round(3)

## Question 2: The substitution network

From the *X instead of Y* sentences we build a **second network**, made from text (week 7–8): nodes are ingredients, and a directed link Y → X means *a reviewer replaced Y with X*, weighted by how many times. We only use substitutions where Y is really in the recipe (the automatic check from the test run).

In [ ]:
import networkx as nx

good = both[both["y_in_recipe"]]
counts = good.groupby(["y", "x"]).size()

SG = nx.DiGraph()
for (y, x), n in counts.items():
    SG.add_edge(y, x, weight=n)

print(f"Substitution network: {SG.number_of_nodes():,} ingredients, {SG.number_of_edges():,} links, "
      f"from {len(good):,} substitutions")
replaced = pd.Series(dict(SG.out_degree(weight="weight"))).nlargest(10)
used = pd.Series(dict(SG.in_degree(weight="weight"))).nlargest(10)
print("\nMost often replaced:", ", ".join(f"{k} ({v})" for k, v in replaced.items()))
print("Most often used instead:", ", ".join(f"{k} ({v})" for k, v in used.items()))

### Do substitutes share the same neighbours, without being used together?

Two ingredients can be related in two ways in the recipes:

- **Used together** (complements, e.g. butter and sugar): they appear in the same recipes more often than chance. We measure this with *observed / expected co-occurrence*, the ratio inside PMI (week 6): O/E = n(a,b) · N / (n(a) · n(b)). O/E > 1 means more often together than chance.
- **Same company** (e.g. butter and margarine): they appear *with the same other ingredients*. We measure this as the cosine between their co-occurrence profiles: for each ingredient, the vector of how often it appears with every other ingredient (the distributional hypothesis from week 6, used on ingredients instead of words).

Our expectation: substitutes keep the same company but are rarely used together, because you use one *instead of* the other.

**Null model:** we keep all substitution pairs, but shuffle which ingredient was put in (X) among the pairs. Each ingredient keeps the number of times it is replaced and used, but the pairs become random (the degree-preserving shuffle from week 2).

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
X = mlb.fit_transform(recipes["ingr_clean"])
col = {name: k for k, name in enumerate(mlb.classes_)}
n_recipes = len(recipes)
df = X.sum(axis=0)
C = (X.T @ X).astype(float)          # C[a, b] = number of recipes with both a and b
np.fill_diagonal(C, 0)
profile = C / np.maximum(np.linalg.norm(C, axis=1, keepdims=True), 1e-12)

pairs = counts.reset_index()[["y", "x"]]
pairs = pairs[pairs["y"].isin(col) & pairs["x"].isin(col)]


def pair_measures(y_list, x_list):
    a = np.array([col[v] for v in y_list])
    b = np.array([col[v] for v in x_list])
    a, b = a[a != b], b[a != b]          # a shuffle can pair an ingredient with itself; drop those
    oe = C[a, b] * n_recipes / (df[a] * df[b])
    same_company = np.sum(profile[a] * profile[b], axis=1)
    return oe, same_company


real_oe, real_company = pair_measures(pairs["y"], pairs["x"])

null_company, null_never, null_more = [], [], []
for _ in range(100):
    oe, company = pair_measures(pairs["y"], rng.permutation(pairs["x"].to_numpy()))
    null_company.append(np.median(company))
    null_never.append((oe == 0).mean())
    null_more.append((oe > 1).mean())

print(f"Substitution pairs used: {len(pairs):,}\n")
print(f"Same company (profile cosine), median:  real {np.median(real_company):.2f}, "
      f"shuffled {np.mean(null_company):.2f} ± {np.std(null_company):.2f}")
print(f"Never in the same recipe:               real {(real_oe == 0).mean():.0%}, "
      f"shuffled {np.mean(null_never):.0%} ± {np.std(null_never):.1%}")
print(f"In the same recipe more than by chance: real {(real_oe > 1).mean():.0%}, "
      f"shuffled {np.mean(null_more):.0%} ± {np.std(null_more):.1%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
shuffled_oe, shuffled_company = pair_measures(pairs["y"], rng.permutation(pairs["x"].to_numpy()))

bins = np.linspace(0, 1, 30)
ax1.hist(shuffled_company, bins=bins, alpha=0.6, color="#999999", label="Shuffled pairs")
ax1.hist(real_company, bins=bins, alpha=0.7, color="#0072B2", label="Substitution pairs")
ax1.set_xlabel("Same company: cosine of co-occurrence profiles")
ax1.set_ylabel("Number of pairs")
ax1.legend(frameon=False)

bins = np.logspace(-2, 1.5, 30)
ax2.hist(shuffled_oe[shuffled_oe > 0], bins=bins, alpha=0.6, color="#999999", label="Shuffled pairs")
ax2.hist(real_oe[real_oe > 0], bins=bins, alpha=0.7, color="#0072B2", label="Substitution pairs")
ax2.axvline(1, color="#D55E00", ls="--", label="As often as chance")
ax2.set_xscale("log")
ax2.set_xlabel("Used together: observed / expected co-occurrence")
ax2.legend(frameon=False)
plt.suptitle("Substitutes keep the same company — but are they used together?")
plt.tight_layout()
plt.show()

### Drawing the substitution network

The most common substitutions (at least 8 times), following the drawing rules from week 4: force layout, node size = how often the ingredient is involved in a substitution, arrow Y → X = *Y replaced by X*, link width = how often.

In [ ]:
strong = nx.DiGraph([(y, x, d) for y, x, d in SG.edges(data=True) if d["weight"] >= 8])
involved = dict(SG.degree(weight="weight"))
pos = nx.spring_layout(strong.to_undirected(), seed=3, k=0.9)

fig, ax = plt.subplots(figsize=(13, 10))
nx.draw_networkx_edges(strong, pos, ax=ax, width=[0.3 + d["weight"] / 25 for _, _, d in strong.edges(data=True)],
                       alpha=0.5, edge_color="#555555", arrows=True, arrowsize=8, connectionstyle="arc3,rad=0.1")
nx.draw_networkx_nodes(strong, pos, ax=ax, node_size=[20 + involved[n] * 1.5 for n in strong],
                       node_color="#E69F00", edgecolors="white", linewidths=0.5)
nx.draw_networkx_labels(strong, pos, ax=ax, font_size=8)
ax.set_title(f"What home cooks swap: {strong.number_of_edges()} substitutions mentioned at least 8 times in the reviews")
ax.axis("off")
plt.tight_layout()
plt.show()

## Question 3: Does the recipe *as cooked* move in the recipe network?

We now go back to the recipe network from `project_Rakel.ipynb`. For every recipe with at least one *X instead of Y*, we take its most common substitution and make the recipe **as cooked**: Y is replaced by X in its ingredient list. Then we build the full network again in exactly the same way (TF-IDF, cosine similarity between all recipes), so the links change a little.

**Do the changed recipes move away from their community?** We keep the 13 communities from the recipe network (running Louvain again would not work: it gives slightly different communities every time, NMI 0.70 between runs, which would hide the small changes). For every changed recipe we measure:

- **own share**: the share of its link weight that goes to recipes in its own community;
- **moved**: whether the community it has most link weight to is a different one than before the change.

Every change moves a recipe a little, because Y was one of the ingredients that placed it in its community. So we compare with a **null model**: the same recipes, the same Y, but X is shuffled between the recipes (the same replacement ingredients, given at random), 20 times. If home cooks stay in the same culinary neighbourhood, the real changes should move the recipes much less than the random ones.

In [ ]:
from sklearn.preprocessing import normalize

order = recipes["id"].to_numpy()
community_of = dict(zip(communities["id"], communities["community"]))
labels = np.array([community_of[r] for r in order])

# the most common substitution for each recipe (Y must be in the recipe)
top_swap = (good.groupby(["recipe_id", "y", "x"]).size().rename("n").reset_index()
                .sort_values("n", ascending=False).drop_duplicates("recipe_id"))
changed = top_swap["recipe_id"].to_numpy()
print(f"Recipes with at least one substitution: {len(changed):,} of {len(recipes):,}")


def apply_swaps(swaps):
    """Ingredient lists 'as cooked': in each recipe, Y is replaced by X."""
    new = dict(zip(recipes["id"], recipes["ingr_clean"]))
    for r, y, x in zip(swaps["recipe_id"], swaps["y"], swaps["x"]):
        new[r] = sorted(set(new[r]) - {y} | {x})
    return [new[r] for r in order]


def community_weights(ingredient_lists):
    """Build the full network as in project_Rakel.ipynb (TF-IDF, cosine) and return,
    for the changed recipes, their own share and the community they have most link weight to."""
    B = MultiLabelBinarizer().fit_transform(ingredient_lists).astype(np.float32)
    V = normalize(B * np.log(len(B) / B.sum(axis=0)))
    S = V @ V.T
    np.fill_diagonal(S, 0)
    W = pd.DataFrame(S, index=order).T.groupby(labels).sum().T.loc[changed]   # link weight to each community
    own = np.array([community_of[r] for r in changed])
    return pd.DataFrame({"own share": W.to_numpy()[np.arange(len(W)), own] / W.sum(axis=1),
                         "top": W.idxmax(axis=1)})


def summary(res):
    return pd.Series({"own share": res["own share"].mean(), "moved": (res["top"] != written["top"]).mean()})


written = community_weights(recipes["ingr_clean"])
cooked = community_weights(apply_swaps(top_swap))

rng = np.random.default_rng(0)
random_runs = []
for _ in range(20):
    shuffled = top_swap.assign(x=rng.permutation(top_swap["x"].to_numpy()))
    shuffled = shuffled[shuffled["x"] != shuffled["y"]]
    random_runs.append(summary(community_weights(apply_swaps(shuffled))))
random_runs = pd.DataFrame(random_runs)

result = pd.DataFrame({"as written": summary(written), "as cooked": summary(cooked),
                       "random (mean)": random_runs.mean(), "random (sd)": random_runs.std()})
print(result.round(4))
drop_cooked = summary(written)["own share"] - summary(cooked)["own share"]
drop_random = summary(written)["own share"] - random_runs["own share"].mean()
print(f"\nThe real changes move the recipes {drop_cooked / drop_random:.0%} as far as random changes "
      f"(z = {(summary(cooked)['own share'] - random_runs['own share'].mean()) / random_runs['own share'].std():.0f})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, title in [(ax1, "own share", "Share of link weight to own community"),
                       (ax2, "moved", "Share of changed recipes that move community")]:
    ax.hist(random_runs[col], bins=10, color="#999999", label="random substitutions (20 runs)")
    ax.axvline(result.loc[col, "as written"], color="#0072B2", lw=2, label="as written")
    ax.axvline(result.loc[col, "as cooked"], color="#D55E00", lw=2, label="as cooked (real substitutions)")
    ax.set_title(title)
    ax.set_ylabel("runs")
ax1.legend(fontsize=8)
fig.suptitle(f"The {len(changed):,} recipes with a substitution: real changes keep them closer to their community than random ones")
plt.tight_layout()
plt.show()

**What we see:**

- **Every change moves the recipes a little**, but the real changes much less than random ones. The share of link weight to the own community drops from 0.372 to 0.360 with the real substitutions, against 0.341 ± 0.001 with random ones: the real changes move the recipes only **37 % as far** as random changes (z ≈ 34).
- **9.4 % of the changed recipes get a new strongest community** with the real substitutions, against 10.4 % ± 0.4 % with random ones (all 20 random runs were higher). This difference is much smaller: many recipes lie between two communities (the full network is dense, and Louvain communities are not sharp), so even one changed ingredient can tip them over.
- So home cooks change the recipes, but mostly **inside the same part of the recipe network**: they swap butter for margarine and beef for turkey, which are used in the same kind of recipes, not ingredients from another kitchen.

### What we found

**Question 1: Do people change baking recipes less?**

- **The differences are real.** The spread of the community means is 0.043, against 0.008 ± 0.002 in 1,000 shuffles (none of them came close, p ≈ 0.001).
- **Yeast bread is changed least** (20.4 %, z ≈ −11): far below the 95 % range of the shuffles (25.5–27.8 %). **Soups and stews (35 %, z ≈ +12) and Mexican food (33 %, z ≈ +11) are changed most.**
- **Sweet baking (cookies, cakes, chocolate) is also below chance (24.8 %, z ≈ −5)**, but much less than bread. So *baking is chemistry* holds most clearly for bread.
- **It is not the boring explanations:**
  - With the strict measure (no *added*), the ordering of the communities is almost the same (Spearman 0.86), and bread is still among the lowest.
  - Without the 281 users with more than 100 reviews, nothing changes (Spearman 0.99).
  - With a broad measure that also catches everyday phrases (*I used ..., I didn't use ..., doubled, skipped*), the ordering is almost the same (Spearman 0.93): yeast bread is still among the lowest (41 %) and soups (55 %) and Mexican food (54 %) are still the highest.
  - Recipes with more ingredients *are* changed more (Spearman 0.36), so this matters. But for every number of ingredients, yeast bread is changed less than other recipes with the same number of ingredients (e.g. 8–9 ingredients: 22 % vs 28 %).

**Question 2: What do home cooks swap?**

- The substitution network has 942 ingredients and 2,701 different substitutions (from 7,757 mentions). Most often replaced: water, butter, beef, milk, margarine, sugar. Most often put in: butter, olive oil, milk, turkey, chicken broth.
- In the drawing you can see *kitchens*: a baking-fat cluster (butter, margarine, shortening, olive oil, oil), a dairy cluster (milk, cream, sour cream, yogurt), a meat cluster (beef, turkey, chicken, pork), nuts (pecans, walnuts, almonds), and water being replaced by broth or milk (adding flavour).
- **Substitutes keep the same company:** the cosine of their co-occurrence profiles is 0.68 against 0.52 for shuffled pairs. So the distributional idea from week 6 works for ingredients: *you shall know an ingredient by the ingredients it keeps*.
- **But our expectation that substitutes are rarely used together was wrong.** They are *more* often in the same recipe than shuffled pairs: 36 % of the substitution pairs appear together more often than chance (shuffled: 24 % ± 1 %), and only 52 % never appear in the same recipe (shuffled: 61 %). Butter and margarine, garlic and garlic powder often appear in the same kind of recipe, or even in the same recipe. Substitutes are ingredients from the same kitchen, not opposites.

**Question 3: Does the recipe *as cooked* move in the recipe network?**

- We rebuilt the full recipe network with the most common substitution applied to each of the 2,654 recipes that have one. The real changes move the recipes only 37 % as far from their community as random changes (share of link weight to the own community: 0.372 → 0.360, random 0.341 ± 0.001, z ≈ 34). The share of recipes that get a new strongest community is only a little lower than with random changes (9.4 % vs 10.4 % ± 0.4 %).

**Validation (hand labels, see below):**

- *Mentions a change*: in 100 hand-labelled reviews, 48 % describe a change. Our main measure has precision 96 % but recall 50 %: when it says *change* it is almost always right, but it misses half of the changes. So the true share of reviews that describe a change is about half, not a quarter. The broad measure has precision 91 % and recall 83% (but its phrases were chosen after reading this sample, so its recall is optimistic). Because the ordering of the communities is the same with both measures, the missed changes do not explain the result.
- *X instead of Y*: in 50 hand-labelled substitutions, Y (the replaced ingredient) is correct in 96 % and X (the new one) in 88 %, both in 86 %. Most errors are partial matches of ingredients that are not in our vocabulary (*potato flour → potato*, *chocolate covered raisins → chocolate*). The sentence order *"instead of Y, I used X"* (16 % of the *instead of* clauses) is not caught, so the substitution network is incomplete.


## Hand-labelled samples (validation)

Our two measurements (*mentions a change* and *X instead of Y*) must be checked against what a person reads in the review (week 7–8). This cell makes two Excel files in `labels/`, which we fill in by hand (Excel files, because a Danish Excel cannot read comma-separated CSV files):

1. **`labels/change_sample.xlsx`**: 100 random reviews. In the column `label_change`, write **1** if the review describes a change the cook made to the recipe (added, left out, replaced, more/less of something), and **0** if not. The automatic answer is *not* in the file, so it cannot influence us.
2. **`labels/substitution_sample.xlsx`**: 50 random extracted substitutions, with the sentence around *instead of* in `context`. In `x_ok` and `y_ok`, write **1** if X (the new ingredient) or Y (the replaced ingredient) is correct, and **0** if not. If it is wrong, write the right ingredient in `x_should_be` / `y_should_be`.

The files are only created if they do not exist yet, so running the notebook again never overwrites our labels. `labels/` is on GitHub, so we can share the work.

In [ ]:
import os


def save_for_labelling(table, path, widths):
    """Save as an Excel file with wide, wrapped columns, so it is easy to read and fill in."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        table.to_excel(writer, index=False)
        sheet = writer.sheets["Sheet1"]
        for col, width in zip("ABCDEFGHIJ", widths):
            sheet.column_dimensions[col].width = width
        for row in sheet.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = cell.alignment.copy(wrap_text=True, vertical="top")
        sheet.freeze_panes = "A2"
    print("Created", path)


os.makedirs("labels", exist_ok=True)

if not os.path.exists("labels/change_sample.xlsx"):
    sample = reviews.sample(100, random_state=1)[["user_id", "recipe_id", "text"]]
    sample["label_change"] = ""
    save_for_labelling(sample, "labels/change_sample.xlsx", [12, 10, 100, 14])

if not os.path.exists("labels/substitution_sample.xlsx"):
    sample = both.sample(50, random_state=1)[["recipe_id", "context", "x", "y"]].copy()
    sample["x_ok"], sample["y_ok"], sample["x_should_be"], sample["y_should_be"] = "", "", "", ""
    save_for_labelling(sample, "labels/substitution_sample.xlsx", [10, 70, 18, 18, 7, 7, 18, 18])

When the files are filled in, this cell compares our labels with the automatic measurements.

In [ ]:
change = pd.read_excel("labels/change_sample.xlsx")
if change["label_change"].notna().all():
    change = change.merge(reviews[["user_id", "recipe_id", "mentions_change", "mentions_change_broad"]],
                          on=["user_id", "recipe_id"])
    print(f"Hand labels: {(change['label_change'] == 1).mean():.0%} of the reviews describe a change")
    for column in ["mentions_change", "mentions_change_broad"]:
        tp = (change[column] & (change["label_change"] == 1)).sum()
        print(f"{column:22s} precision {tp / change[column].sum():.0%} (of the reviews we flag, how many really describe a change), "
              f"recall {tp / (change['label_change'] == 1).sum():.0%} (of the reviews that describe a change, how many we flag)")
else:
    print("labels/change_sample.xlsx is not filled in yet")

subs_labels = pd.read_excel("labels/substitution_sample.xlsx")
if subs_labels[["x_ok", "y_ok"]].notna().all().all():
    print(f"Substitutions: X correct {subs_labels['x_ok'].mean():.0%}, Y correct {subs_labels['y_ok'].mean():.0%}, "
          f"both correct {((subs_labels['x_ok'] == 1) & (subs_labels['y_ok'] == 1)).mean():.0%}")
else:
    print("labels/substitution_sample.xlsx is not filled in yet")